# DITSB-v2 大语言模型 Google Colab 训练一键脚本

使用前请确保您已在顶部菜单栏 **代码执行程序 (Runtime)** -> **更改代码执行程序类型 (Change runtime type)** 中选择了 **GPU (T4/V100/A100)**。

## 1. 挂载 Google Drive
挂载网盘以便持久化保存训练好的模型权重 (Checkpoints)，防止 Colab 断线导致数据丢失。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 下载 DITSB 源代碼与安装依赖
从 GitHub 拉取代码储备并安装所需的所有 Python 库。

In [ ]:
# 请将下面的链接替换为您的实际 GitHub 仓库链接，如果代码是私有也可以考虑通过 Google Drive 复制压缩包解压。
!git clone https://github.com/serh1m/DITSB.git /content/DITSB

import os
os.chdir('/content/DITSB')
print("当前工作目录:", os.getcwd())

# 安装依赖
!pip install datasets transformers accelerate pyyaml wandb tqdm
if os.path.exists('requirements.txt'):
    !pip install -r requirements.txt

## 3. 准备与下载训练数据
数据将下载到 Colab 本地 `/content/dataset` 目录 (高速 SSD)，**不要**存到网盘里。

In [ ]:
# 执行数据集准备脚本 (假设您默认使用的脚本支持这些参数)
!python llm_training/prepare_data.py --output_dir=/content/dataset --max_samples=50000

## 4. 自动修改配置
将 `config_7b.yaml` 里的数据路径指向本地高速目录，并将存档点 (Checkpoint) 指向 Google Drive。

In [ ]:
import yaml

config_path = 'llm_training/config_7b.yaml'

# 读取并修改关键参数
try:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # 设定数据路径 (必须在 /content/ 下)
    config['data']['dataset_dir'] = '/content/dataset'
    
    # 设定 Checkpoint 存储路径 (必须在 /content/drive 下)
    config['training']['checkpoint_dir'] = '/content/drive/MyDrive/DITSB_checkpoints'
    
    # 根据 Colab 的 GPU 能力降低 Batch Size
    config['training']['batch_size'] = 2  # 对于Colab T4，可能需要降低
    config['training']['grad_accum_steps'] = 16 # 保持等效大 Batch Size
    
    # 调整单卡配置
    config['hardware']['num_nodes'] = 1
    config['hardware']['gpus_per_node'] = 1

    with open(config_path, 'w') as f:
        yaml.dump(config, f)
        
    print("✅ 配置文件已成功更新以适应 Colab 环境！")
    print("Checkpoint 将保存在: /content/drive/MyDrive/DITSB_checkpoints")
except FileNotFoundError:
    print(f"⚠️ 找不到配置文件: {config_path}，请确认路径。")

## 5. 性能预估 (可选)
先预估跑一下，看看当前 GPU 下训练时长需要多久。

In [ ]:
!python llm_training/predict_performance.py --config llm_training/config_7b.yaml

## 6. 🔥 正式启动大模型训练
点击运行！随时留意网盘空间是否充足。

In [ ]:
import os

# 确保 Checkpoint 存儲目录存在
os.makedirs('/content/drive/MyDrive/DITSB_checkpoints', exist_ok=True)

# 启动训练脚本
!python llm_training/train_llm.py --config llm_training/config_7b.yaml